# Citi Velocity through the Excel add-in

Pulls a **swap curve**, a **swaption-cube slice** and an **intraday UST series**
from Citi Velocity, then strips a curve locally in both rateslib and QuantLib and
shows that the published quote and the repriced curve agree.

## Before you run this

This notebook drives **your own Excel process** over COM. There is no headless
path: the Velocity login is gated on the ribbon loading, and spawned Excel
instances never register the `CV*` UDFs.

* Excel must be open and **signed in to Velocity**.
* After an Excel restart the login takes **~13 minutes** and logs nothing in
  between - if `connect()` fails immediately after a restart, wait, do not retry
  in a loop.
* Never kill a cell mid-call. That wedges Excel's OLE server for ~15 minutes.

If no signed-in add-in is reachable, the next cell falls back to a **synthetic
fake** so the whole notebook still runs end to end. Every output is then clearly
marked `SYNTHETIC` and none of the numbers mean anything about the market.

In [ ]:
from __future__ import annotations

import datetime
import warnings

import numpy as np
import pandas as pd

from MDP.CitiVelocityExcel import CitiVelocityExcelClient, CitiVeloCatalog, CitiVeloTagCache
from MDP.CitiVelocityExcel import tags as T
from MDP.CitiVelocityExcel.catalog import tenor_years
from MDP.CitiVelocityExcel.errors import CitiVelocityError
from MDP.CitiVelocityExcel.mdp import CitiVelocityMDP
from MDP.CitiVelocityExcel.quotes import CitiVeloQuotes

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 50)

## 1. Connect (or fall back to the fake)

In [ ]:
LIVE = True
try:
    client = CitiVelocityExcelClient.connect(attempts=1, readiness_timeout=60.0)
    print("connected to a signed-in Citi Velocity add-in")
except Exception as exc:
    LIVE = False
    print(f"NO LIVE ADD-IN ({type(exc).__name__}: {exc})")
    print(">>> falling back to the synthetic fake - every number below is SYNTHETIC <<<")

    from MDP.CitiVelocityExcel.testing import FakeExcelApp, FakeVelocityData

    _index = pd.bdate_range("2025-08-01", "2026-08-04")
    _series = {}
    for _tag in T.ois_par_grid("USD_SOFR"):
        _y = tenor_years(_tag.rsplit(".", 1)[-1])
        _level = 3.60 + 0.90 * (1.0 - np.exp(-_y / 3.0))
        _series[_tag] = pd.Series(_level + 0.05 * np.sin(np.arange(len(_index)) / 40.0), index=_index)
    for (_e, _t), _tag in T.vol_atm_grid(
        "USD", expiries=("1M", "3M", "1Y", "5Y"), tenors=("2Y", "5Y", "10Y", "30Y")
    ).items():
        _base = 55.0 + 60.0 * np.exp(-tenor_years(_t) / 8.0) + 20.0 * np.exp(-tenor_years(_e) / 2.0)
        _series[_tag] = pd.Series(_base + 0.5 * np.cos(np.arange(len(_index)) / 30.0), index=_index)
    _minutes = pd.date_range("2026-08-03 09:00", "2026-08-04 16:00", freq="1min")
    _series[T.tsy_otr("10Y")] = pd.Series(
        4.20 + 0.02 * np.cumsum(np.random.default_rng(0).normal(0, 0.01, len(_minutes))), index=_minutes
    )
    client = CitiVelocityExcelClient(app=FakeExcelApp(FakeVelocityData(series=_series), pending_reads=1),
                                     drain_seconds=0.0)

BANNER = "" if LIVE else "  [SYNTHETIC]"

## 2. The catalog is a grammar, not a tag list

Walking to every leaf is infeasible - `RATES.VOL` alone is ~250,000 tags across 11
currencies. The catalog stores structure plus per-branch grammar; you generate the
tag you want and validate it on demand.

Note the real index tokens: `EUR_EUROSTR` (not ESTR), `USD_FEDFUND` (singular),
`JPY_TONAR_JSCC` / `JPY_TONAR_LCH` (CCP-split). These are not guessable - an
earlier generate-and-test pass got both of the biggest currencies wrong.

In [ ]:
cat = CitiVeloCatalog.default()
print(f"{len(cat.families())} RATES.* families")
print("OIS curves:", ", ".join(cat.options("RATES.OIS")))
print()
print("sub-types:", cat.options("RATES.OIS.USD_SOFR"))
print("PAR axis :", len(cat.tenors("RATES.OIS.USD_SOFR.PAR")), "tenors,",
      cat.tenors("RATES.OIS.USD_SOFR.PAR")[:6], "...")
print()
print("depth varies PER BRANCH - this is why a level-pooled generator fails:")
print("  ATM_RFR.NORMAL ->", cat.options("RATES.VOL.USD.ATM_RFR.NORMAL"))
print("  ATM_RFR.BLACK  ->", cat.options("RATES.VOL.USD.ATM_RFR.BLACK")[:5], "... (straight to expiry)")
print()
print("  vol_atm NORMAL:", T.vol_atm("USD", "1Y", "10Y"))
print("  vol_atm BLACK :", T.vol_atm("USD", "1Y", "10Y", measure="BLACK"))
print("  vol_otm -25bp :", T.vol_otm("USD", "1Y", "10Y", -25))
print("  vol_otm -25bp (PREMIUM, different spelling):",
      T.vol_otm("USD", "1Y", "10Y", -25, measure="PREMIUM"))

## 3. Validation goes through `CVTSHIST`, never `CVMETADATA`

`CVMETADATA` hard-fails to `#VALUE!` for tags that exist and serve data - it
reports **zero** valid tenors for the whole `SWAP_SPREAD` family while `CVTSHIST`
serves all eleven. And every validation run puts known-good control tags in front,
because two validators in the design session produced confident wrong numbers and
the controls are what caught them.

In [ ]:
if LIVE:
    verdicts = client.validate_with_controls(
        [T.ois_swap_spread("USD_SOFR", t) for t in T.SWAP_SPREAD_LIQUID_TENORS]
    )
    served = sum(1 for v in verdicts.values() if v == "valid")
    print(f"SWAP_SPREAD: {served}/{len(verdicts)} tenors serve through CVTSHIST")
    for tag, verdict in list(verdicts.items())[:5]:
        print(f"  {verdict:<8} {tag}")
else:
    print("skipped: validation needs the live add-in" + BANNER)

## 4. A swap curve - 44 tenors in ONE `CVTSHIST` call

In [ ]:
CURVE = "USD_SOFR"
grid_tags = T.ois_par_grid(CURVE)
grid = client.fetch_frame(grid_tags, "DAILY", period="1Y")
grid.columns = [c.rsplit(".", 1)[-1] for c in grid.columns]
print(f"{grid.shape[0]} rows x {grid.shape[1]} tenors, one CVTSHIST call{BANNER}")
grid.tail(3).iloc[:, ::6]

In [ ]:
latest = grid.iloc[-1]
ax = latest.reset_index(drop=True).plot(
    figsize=(11, 4), marker="o", title=f"{CURVE} par curve, {grid.index[-1]:%Y-%m-%d}{BANNER}"
)
ax.set_xticks(range(len(latest)))
ax.set_xticklabels(latest.index, rotation=90, fontsize=7)
ax.set_ylabel("par rate, %")
ax.grid(alpha=0.3)

## 5. Strip it locally, in both backends

Velocity's own pricers (`CVDSWAP`, `CVCALCDERIVATIVES`, ...) are **not entitled**,
so every curve here is ours. Both backends anchor on the same rolled reference
date, use the same spot lag and pin nodes on the same swap maturities.

In [ ]:
from MDP.CitiVelocityExcel.curves import (
    build_ql_ois_curve,
    build_rl_ois_curve,
    conventions_for,
    par_reprice_errors_bp,
    ql_forward_rate,
    forward_rate,
)

ref_date = grid.index[-1].date()
par_rates = {tenor: float(v) for tenor, v in latest.items() if pd.notna(v)}

rlc = build_rl_ois_curve(par_rates=par_rates, ref_date=ref_date, citi_index=CURVE)
qlc = build_ql_ois_curve(par_rates=par_rates, ref_date=ref_date, citi_index=CURVE)

errors = par_reprice_errors_bp(rlc)
print(f"rateslib reprices its own {len(errors)} inputs to max {errors.abs().max():.2e} bp")
print(f"conventions: {conventions_for(CURVE).provenance}")
print()
print(f"{'forward':<12}{'rateslib':>12}{'QuantLib':>12}{'gap bp':>10}")
for fwd, tenor in (("1Y", "1Y"), ("2Y", "3Y"), ("5Y", "5Y"), ("10Y", "10Y")):
    a = forward_rate(rlc, forward=fwd, tenor=tenor)
    b = ql_forward_rate(qlc, forward=fwd, tenor=tenor)
    print(f"{fwd}x{tenor:<9}{a:>12.6f}{b:>12.6f}{(a - b) * 100:>10.5f}")

## 6. A swaption-cube slice

`RATES.VOL.<ccy>.ATM_RFR...` is already expiry x tenor, i.e. a ready-made surface.
The `_RFR` branches are the live ones; the legacy `ATM`/`OTM` twins account for
every shape failure in the harvest and return no data.

In [ ]:
EXPIRIES = ("1M", "3M", "1Y", "5Y")
TENORS = ("2Y", "5Y", "10Y", "30Y")

atm_tags = T.vol_atm_grid("USD", expiries=EXPIRIES, tenors=TENORS)
vols = client.fetch_frame(list(atm_tags.values()), "DAILY", period="1M")
snapshot = vols.iloc[-1]

surface = pd.DataFrame(
    [[snapshot.get(atm_tags[(e, t)], np.nan) for t in TENORS] for e in EXPIRIES],
    index=EXPIRIES,
    columns=TENORS,
)
print(f"USD ATM normal vol, bp, {vols.index[-1]:%Y-%m-%d}{BANNER}")
surface.round(2)

## 7. An intraday UST series

One minute is the finest historical granularity for **every** family. The desk's
intraday workbook advertises `SE10`, but that describes the streaming feed -
`CVTSHIST` rejects it outright.

In [ ]:
ust_tag = T.tsy_otr("10Y")
intraday = client.fetch_frame([ust_tag], "MI01", period="2D")
print(f"{len(intraday)} one-minute rows{BANNER}")
if not intraday.empty:
    deltas = pd.Series(intraday.index).diff().dropna()
    print(f"finest observed gap: {deltas.min()}")
    intraday.plot(figsize=(11, 3.5), title=f"UST 10Y OTR yield, 1-minute{BANNER}", legend=False, lw=0.8)

from MDP.CitiVelocityExcel.frequencies import normalise_frequency
try:
    normalise_frequency("SE10")
except Exception as exc:
    print("\nSE10:", exc)

## 8. Structures through the timeseries builder, and the fast path

Most Velocity queries **are already a tag**, so repricing them at every timestep
would strip a curve to recover a number the add-in already published. The builder
routes those to one bulk read - and `assert_fast_path_matches` proves the shortcut
agrees with the path it replaces, which is what makes it an optimisation rather
than a silent divergence.

In [ ]:
from Query.CitiVelocity import CitiVeloQuery, CitiVeloStructure, CitiVeloValue
from TB.CitiVelocityTB import CitiVelocityTB

cache = CitiVeloTagCache()
quotes = CitiVeloQuotes(client=client, cache=cache)
tb = CitiVelocityTB(CitiVelocityMDP(quotes=quotes), show_tqdm=False)

queries = [
    CitiVeloQuery(citi_index=CURVE, tenor="10Y", name="10y"),
    CitiVeloQuery(citi_index=CURVE, structure=CitiVeloStructure.CURVE,
                  structure_kwargs={"front_tenor": "2Y", "back_tenor": "10Y"}, name="2s10s"),
    CitiVeloQuery(citi_index=CURVE, structure=CitiVeloStructure.FLY,
                  structure_kwargs={"front_tenor": "2Y", "belly_tenor": "5Y",
                                    "back_tenor": "10Y"}, name="2s5s10s"),
]

plan = tb.plan(queries)
print(plan.summary().to_string(index=False))

end = grid.index[-1].date()
start = end - datetime.timedelta(days=60)
frame = tb.get_timeseries(start, end, queries)
print()
print(f"{frame.shape[0]} points x {frame.shape[1]} columns{BANNER}   (10y in %, spreads in bp)")
frame.tail(3)

In [ ]:
comparison = tb.assert_fast_path_matches(
    end - datetime.timedelta(days=10), end,
    [CitiVeloQuery(citi_index=CURVE, tenor="10Y")],
    model_value=CitiVeloValue.RL_RATE,
    tol=0.01,
    raise_on_breach=False,
)
worst = float(comparison["diff"].abs().max())
print(f"published quote vs locally-stripped curve: max |diff| = {worst:.3e}{BANNER}")
print("(the curve is calibrated to that very quote, so this is solver tolerance)")
comparison.tail(3)

## 9. Cache reuse

The cache is keyed `(tag, freq, price_point)` and serves cached rows, fetching only
the missing span. A fully-cached read never touches Excel at all - which is what
makes a warm-cache backtest runnable without a signed-in add-in.

In [ ]:
calls_before = client.calls
warm = tb.get_timeseries(start, end, queries)
print(f"CV* calls for the repeat run: {client.calls - calls_before}")
print(f"identical result: {frame.equals(warm)}")
print()
print(f"cache root : {cache.base_dir}")
print(f"cached keys: {len(cache.keys())}   ({cache.size_bytes() / 1e6:.2f} MB)")

offline = CitiVeloQuotes(cache=cache, offline=True)
served = offline.frame(grid_tags[:5], "DAILY", start=start, end=end)
print(f"\noffline read with NO client: {served.shape[0]} rows x {served.shape[1]} tags")

## 10. Tidy up

Closes the scratch workbook after a drain pause. Nothing is ever cleared or
deleted: tearing a region down while the add-in's queued `ExcessClr`/`Format`/
`AutoFit` actions are outstanding is itself an `AccessViolation` trigger, and it
takes the whole Excel process with it.

In [ ]:
client.close()
print("scratch workbook closed; your own workbooks were never touched")